In [2]:


import json

with open("../data/results_20250815.json", "r") as f:
    results = json.load(f)


successful = []
failed = []
 
for result in results:
    if result['metadata']['num_chunks_used'] >= 1:
        successful.append(result)
    else:
        failed.append(result)

print(len(successful))
print(len(failed))



14
3


In [ ]:


retrieval_accuracy = []

for result in successful:
    retrieval_number = 0 
    
    expected_chunks = []
    for chunk in result['expected_chunks']:
        if type(chunk['chunk_id']) == list:
            expected_chunks.extend(chunk['chunk_id'])
        else:
            expected_chunks.append(chunk['chunk_id'])



    for chunk in result['chunks']:
        if chunk in expected_chunks:
            retrieval_number += 1
            
    retrieval_accuracy.append(retrieval_number / len(expected_chunks))
    
print(sum(retrieval_accuracy) / len(retrieval_accuracy))



0.8996598639455782


In [21]:
from decompose_string_tool import decompose_string 




retrieval_accuracy = []

for result in successful:
    retrieval_number = 0 
    
    expected_chunks = []
    for chunk in result['expected_chunks']:
        if type(chunk['chunk_id']) == list:
            expected_chunks.extend(chunk['chunk_id'])
        else:
            expected_chunks.append(chunk['chunk_id'])

    chunk_analysis = decompose_string(result['xyz_answer']) 
    chunk_used_index = []
    for chunk_a in chunk_analysis:
        chunk_index = chunk_a['references']
        if chunk_index not in chunk_used_index:
            for number in chunk_index:
                chunk_used_index.append(number-1)
    chunk_used_id = [result['chunks'][index] for index in chunk_used_index]
    # print(chunk_used_id)
    # print(expected_chunks)
    for chunk in expected_chunks:
        if chunk in chunk_used_id:
            retrieval_number += 1
            break
            
    retrieval_accuracy.append(retrieval_number / len(expected_chunks))
    
print(retrieval_accuracy)

print(sum(retrieval_accuracy) / len(retrieval_accuracy))


[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.3333333333333333, 0.3333333333333333, 0.5, 0.2, 0.5, 0.14285714285714285, 0.0, 0.5]
0.6078231292517007


In [6]:
from deep_eval import DeepEval
from decompose_string_tool import decompose_string 
import json

all_data = {}
with open("../experiments_docs_processed/paper_set_1/knowledge_rag_results.json", "r") as f:
    chunks = json.load(f) 
    chunks = chunks['chunks']
    chunks = {chunk['chunk_id']: chunk for chunk in chunks}

with open("../data/results_20250815.json", "r") as f:
    results = json.load(f)
        
            


In [9]:
all_evaluated = []

for result in results:
    input_question = result['question']
    
    input_chunks = result['chunks']
    
    chunk_analysis = decompose_string(result['xyz_answer']) 

    for information in chunk_analysis:
        content = information['content']
        references = information['references']
        try:
            chunk_ids_local = [input_chunks[index-1] for index in references]
            chunk_information = [chunks[chunk_id] for chunk_id in chunk_ids_local]
        except:
            print(f"References: {references}")
            print(f"Input Chunks number: {len(input_chunks)}")
            continue
        
        all_reference_text = []
        for index, chunk in enumerate(chunk_information):
            local_reference_text = ""
            local_reference_text += f"Summary: {chunk.get('summary', '')}\n"
            local_reference_text += f"Insights: {chunk.get('insights', '')}\n"
            local_reference_text += f"OriginalContent: {chunk.get('chunk_markdown_content', '')}\n\n"
            all_reference_text.append(local_reference_text)
        
        all_evaluated.append({
            "question": input_question,
            "content": content,
            "retrieval_context": all_reference_text,
        })
        
        # eval=DeepEval(  # 创建测试用例
        #     question=input_question, 
        #     actual_output=content,
        #     retrieval_context=all_reference_text,
        #     # expected_output=result['xyz_answer']
        # )
        # result = eval.faithfulness()
        
        # print("--------------------------------")
        # print(f"question: {input_question}")
        # print(f"content: {content}")
        # print(f"retrieval_context: {all_reference_text}")
        # print(f"faithfulness: {result}")
        
        
        

References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0
References: [1]
Input Chunks number: 0


In [11]:
import json

with open("../data/metric_0815.json", "w") as f:
    json.dump(all_evaluated, f, indent=4)

In [2]:
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, ContextualRelevancyMetric, ContextualPrecisionMetric, ContextualRecallMetric  # 导入评估器  

from deepeval.test_case import LLMTestCase  # 导入测试用例 
metric= FaithfulnessMetric(  # 创建忠实度评估器
    threshold=0.7,
    model="gpt-4.1",
    include_reason=True,
    # verbose_mode=True
)
test_case = LLMTestCase(  # 创建测试用例
    input="Does the success of chain-of-thought prompting depend on a particular linguistic style?",
    retrieval_context=["Summary: This chunk examines the robustness of chain-of-thought prompting in language models, specifically its sensitivity to differences in annotators, linguistic styles, prompt exemplars, and prompt ordering. Through experiments with LaMDA 137B on GSM8K and MAWPS, and using prompts crafted by multiple annotators and randomly sampled from the GSM8K dataset, the study finds that, while some variance exists, all chain-of-thought prompt sets substantially outperform standard prompting. Robustness extends to exemplar source, order, and quantity, suggesting that chain-of-thought gains do not hinge on prompt idiosyncrasies.\nInsights: ['Chain-of-thought prompting for arithmetic reasoning remains robust despite significant variance in prompt linguistic style, as shown with different annotators.', 'All tested chain-of-thought prompts, regardless of annotator, outperform standard prompting by a large margin in LaMDA 137B experiments on GSM8K and MAWPS benchmarks.', 'Prompting with chains of thought written by independent annotators (not co-authors) also leads to substantial gains, indicating method robustness to writer background.', 'Concise chains of thought, inspired by other solution styles, are similarly effective, showing flexibility in prompt elaboration.', 'Randomly sampled exemplars from GSM8K, which already include reasoning steps, are as effective as carefully written ones, affirming robustness to exemplar source.', 'Chain-of-thought prompting performance is not overly sensitive to the specific examples used in the few-shot prompt, expanding applicability.', 'The robustness of chain-of-thought prompting extends to variations in exemplar order and the number of exemplars, as evidenced by reference to Appendix A.2.', 'Variation in performance across different prompts is expected for exemplar-based prompting, but does not undermine the consistent gains over standard methods.', 'The effective use of chain-of-thought prompting does not require tuning for linguistic style or strict adherence to a solution template.', 'The success of chain-of-thought prompting holds across both manually composed and dataset-derived exemplars.', 'Robustness findings hold for arithmetic reasoning tasks; similar results in other reasoning domains may require further confirmation.', 'Experimental results support that chain-of-thought prompting is not fragile to prompt formulation or author but is a generally dependable improvement technique.']\nOriginalContent: 3.4 Robustness of Chain of Thought\nSensitivity to exemplars is a key consideration of prompting approaches—for instance, varying the permutation of few-shot exemplars can cause the accuracy of GPT-3 on SST-2 to range from near chance $(54.3\\%)$ to near state of the art $(93.4\\%)$ (Zhao et al., 2021). In this final subsection, we evaluate robustness to chains of thought written by different annotators. In addition to the results above, which used chains of thought written by an Annotator A, two other co-authors of this paper (Annotators B and C) independently wrote chains of thought for the same few-shot exemplars (shown in Appendix H). Annotator A also wrote another chain of thought that was more concise than the original, following the style of solutions given in Cobbe et al. (2021).1  \nFigure 6 shows these results for LaMDA 137B on GSM8K and MAWPS (ablation results for other datasets are given in Appendix Table 6 / Table 7). Although there is variance among different chain of thought annotations, as would be expected when using exemplar-based prompting (Le Scao and Rush, 2021; Reynolds and McDonell, 2021; Zhao et al., 2021), all sets of chain of thought prompts outperform the standard baseline by a large margin. This result implies that successful use of chain of thought does not depend on a particular linguistic style.  \n![](https://netmind-public-files.s3.us-west-2.amazonaws.com/8bdbfba71f2f46189f05ddd62ebe2b76/7ec08f4391c9c20cbd82c880587e0ee7cc74ee04caecf44662ef2535468cc7fb.jpg)\nFigure 6: Chain-of-thought prompting has variance for different prompt examples (as expected) but outperforms standard prompting for various annotators as well as for different exemplars.  \nTo confirm that successful chain-of-thought prompting\nworks for other sets of exemplars, we also run experiments\nwith three sets of eight exemplars randomly sampled from the GSM8K training set, an independent  \nsource (examples in this dataset already included reasoning steps like a chain of thought).2 Figure 6 shows that these prompts performed comparably with our manually written exemplars, also substantially outperforming standard prompting.  \nIn addition to robustness to annotators, independently-written chains of thought, different exemplars, and various language models, we also find that chain-of-thought prompting for arithmetic reasoning is robust to different exemplar orders and varying numbers of exemplars (see Appendix A.2).\n\n"],
    actual_output=""" 
Chain-of-thought (CoT) prompting does **not** depend on a particular linguistic style to be effective. Instead, it is robust across different annotators, writing styles, and prompt variations, though careful prompt engineering can still influence performance in some tasks.

---

## Main Findings

### 1. Robustness to Linguistic Style and Annotator Variation

- Chain-of-thought prompting consistently outperforms standard prompting across a wide range of tasks, regardless of the linguistic style or background of the annotator who writes the reasoning steps.    
"""
)
metric.measure(test_case)  # 评估忠实度
print(metric.score)  # 存储忠实度得分

Output()

0.75


In [4]:
metric.reason

'The score is 0.75 because the actual output overgeneralizes the evidence, claiming chain-of-thought prompting outperforms standard prompting across a wide range of tasks, while the retrieval context only supports this for arithmetic reasoning tasks.'

In [12]:
len(all_evaluated)

289

In [4]:
import json 

with open("metric_0815_faithfulness_result.json", "r") as f:
    data = json.load(f)

print(data[0].keys())
print(data[0]["faithfulness"])

dict_keys(['question', 'content', 'retrieval_context', 'faithfulness'])
{'score': 1.0, 'reason': 'Great job! There are no contradictions, so the actual output is fully faithful to the retrieval context.'}


In [10]:
scores= 0
for score in not_sure_result:
    scores += score ["faithfulness"]['score']

In [12]:
274/(269+13)

0.9716312056737588

In [6]:
fail_result = []
good_result = [] 
not_sure_result = []


for faithfulness_data in data: 
    
    if not faithfulness_data["retrieval_context"]:
        fail_result.append(faithfulness_data)
    elif faithfulness_data["faithfulness"]['score'] != 1:
        not_sure_result.append(faithfulness_data)
    else:
        good_result.append(faithfulness_data['faithfulness']['score'])

print(len(fail_result), len(good_result), len(not_sure_result))
    

7 269 13


In [8]:
for not_sure_data in not_sure_result:
    # print(not_sure_data["retrieval_context"])
    print(not_sure_data["faithfulness"])
    # print(not_sure_data["reason"])
    print("-"*100)

{'score': 0.5, 'reason': "The score is 0.50 because the actual output incorrectly claims that the Wikipedia search tool provides up-to-date or detailed content, while the retrieval context only mentions that it returns short text snippets relevant to a search term and does not state anything about supplementing the model's knowledge with up-to-date content."}
----------------------------------------------------------------------------------------------------
{'score': 0.6666666666666666, 'reason': 'The score is 0.67 because the actual output incorrectly claims that QLORA matches full 16-bit finetuning performance at 33B and 65B scales, while the retrieval context clarifies that this equivalence was not established due to resource constraints.'}
----------------------------------------------------------------------------------------------------
{'score': 0.5, 'reason': 'The score is 0.50 because the actual output incorrectly claims that GLUE and Super-NaturalInstructions were main bench

In [15]:
import json 

with open("notebooklm_examples_new_faithfulness.jsonl", "r") as f:
    data = json.load(f)




In [19]:
all_score = 0 
number = 0 

for question in data:
    for chunk in question["chunks"]:
        all_score += chunk["faithfulness"]['score']
        number += 1 

print(all_score/number)

0.9652317880794702


In [ ]:
0.9652317880794702

In [20]:
with open("notebooklm_examples_new_faithfulness.json", "w") as f:
    json.dump(data, f, indent=4)